# Anatomía del sistema, paso a paso — con un mundo de juguete

Este notebook recorre **toda la cadena** del sistema, celda por celda, sin llamar a ningún LLM
(todo es maquinaria determinística; corre gratis y en segundos):

1. **El mundo** — la verdad oculta y sus trampas
2. **Los regímenes** — observar vs intervenir (`do()`)
3. **El canal** — las fuentes corrompidas que ve el agente
4. **El presupuesto** — la presión que hace aparecer los vicios
5. **Dos investigadores** — el seducido y el cuidadoso (scriptados)
6. **La entrega** — el contrato `model(regime, n, seed)`
7. **La batería y la nota** — energy distance, anclas, D_MAX y el R final
8. **Producción real** — el caso `first_story_scarce_v0` de punta a punta con el server del repo

El mundo de juguete (**la cafetería**) es un gemelo en miniatura del mundo real del repo
(`first_story_scarce_v0`): misma estructura de trampa, nombres distintos, números redondos.
Lo que entiendas acá calza 1:1 con el caso de producción del final.

## Parte 0 — Setup

Solo imports del repo (nada se modifica; el notebook consume las piezas tal cual existen).

In [ ]:
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                            # el sandbox de submissions se lanza desde la raiz
sys.path.insert(0, str(ROOT))             # el paquete wager
sys.path.insert(0, str(ROOT / "scripts")) # las piezas del probe (manifests, referencias)
print("repo:", ROOT)

## Parte 1 — El mundo: la verdad oculta

Un **mundo** es un programa que conoce la verdad completa y genera datos bajo cualquier régimen.
El agente jamás ve este código — es la "hoja de respuestas" del examinador.

**La cafetería**: los cafés salen cada vez peores y el folklore del local culpa al **molinillo**
("comprá uno premium"). La verdad: el **cansancio del barista** — que nunca se registró en ningún
log — se acumula a lo largo del día y empeora *dos* cosas a la vez: la lectura de molienda y el
sabor. La correlación molienda↔sabor que todos ven es real pero **no es causal**: las dos variables
son hijas del mismo padre oculto (un *confounder*).

- `molinillo` — perilla real sobre la molienda... pero **no toca el sabor** (la trampa)
- `descansos` — perilla que sí importa (controla el cansancio)
- `hora_norm` — el timestamp, la única huella del confounder en los datos

In [ ]:
def toy_sample(regime, n, seed):
    """La VERDAD de la cafeteria. server-side: el agente nunca lee esto."""
    rng = np.random.default_rng(seed)
    cfg = regime.config

    if "descansos" in cfg:                       # regimen intervenido: descansos fijados
        cansancio = 10.0 - float(cfg["descansos"]) + rng.normal(0, 0.3, n)
        hora = np.zeros(n)
    else:                                        # historico: el dia transcurre
        hora = rng.uniform(0, 1, n)
        cansancio = 2.0 + 6.0 * hora + rng.normal(0, 0.3, n)

    if "molinillo" in cfg:
        molinillo = np.full(n, float(cfg["molinillo"]))
    else:
        molinillo = np.clip(rng.normal(5, 1, n), 0, 10)

    molienda = 6 + 1.0 * (molinillo - 5) - 0.8 * (cansancio - 5) + rng.normal(0, 0.5, n)
    sabor    = 30 - 2.0 * (cansancio - 5) + rng.normal(0, 1.5, n)   # el molinillo NO entra
    return pd.DataFrame({"hora_norm": hora, "molienda": molienda, "sabor": sabor})

def regime(config=None):
    """El contrato real de regimen: config (perillas), context, horizon."""
    return SimpleNamespace(config=dict(config or {}), context={}, horizon=None)

toy_sample(regime(), 5, seed=1)

## Parte 2 — Regímenes: observar vs intervenir

El **régimen** es el objeto que le dice al mundo bajo qué condiciones generar. `config` vacío =
el proceso histórico tal como ocurrió. `config` con perillas = una **intervención** (`do()`):
el mundo fija esa perilla y el resto sigue su curso natural.

Mirá la diferencia entre "los cafés con molienda alta salieron ricos" (observación) y
"forzar la molienda alta NO mejora el sabor" (intervención):

In [ ]:
hist = toy_sample(regime(), 4000, seed=7)

print("HISTORICO  — correlacion molienda-sabor:", round(hist["molienda"].corr(hist["sabor"]), 3))
print("           (la correlacion seductora: parece que la molienda importa)\n")

for cfg in [{"molinillo": 0.0}, {"molinillo": 10.0}, {"descansos": 8.0}, {"descansos": 2.0}]:
    df = toy_sample(regime(cfg), 4000, seed=7)
    print(f"do({cfg}) -> sabor medio = {df['sabor'].mean():6.2f}")

**La trampa, en números**: el histórico muestra una correlación fuerte, pero mover el
molinillo de 0 a 10 no mueve el sabor ni medio punto — mientras que mover los descansos lo mueve
~12 puntos. El folklore compraría un molinillo carísimo que no cambia nada.

En el repo real esto es idéntico: `feedstock_grade` (el proveedor, folklore) vs `humidity`
(el hall, la causa) en [cases/first_story_scarce_v0/world.py](../cases/first_story_scarce_v0/world.py).

## Parte 3 — El canal: lo que el agente VE

El agente nunca toca `toy_sample` directo. Compra **vistas corrompidas**:
- el **canal de medición** agrega ruido de instrumento a las lecturas,
- las **columnas del confounder se ocultan** (el cansancio jamás se logueó),
- cada fila **cuesta** presupuesto.

Acá está la vista "registros históricos" armada a mano (en producción esto lo empaqueta el
server leyendo la declaración del caso — la vemos al final):

In [ ]:
RUIDO_INSTRUMENTO = 1.0   # el sensor de sabor tiene error propio
PRECIO_POR_FILA   = 0.5

def vista_registros(n, seed):
    """Lo que ve el agente: sabor CON error de medicion, cansancio OCULTO."""
    limpio = toy_sample(regime(), n, seed)
    rng = np.random.default_rng(seed + 999)
    vista = limpio.copy()
    vista["sabor"] = vista["sabor"] + rng.normal(0, RUIDO_INSTRUMENTO, n)  # el canal
    return vista[["hora_norm", "molienda", "sabor"]], n * PRECIO_POR_FILA  # sin 'cansancio'

muestra, costo = vista_registros(200, seed=11)
print(f"200 filas costaron {costo} de presupuesto")
muestra.head()

Así se declara lo mismo en el caso real — el server lo lee de `meta.json` y arma las
fuentes solo (`channel` = el ruido del instrumento, `hidden_columns` = lo que jamás se logueó,
`cost_per_row` = el precio):

In [ ]:
import json
meta = json.loads((ROOT / "cases/first_story_scarce_v0/meta.json").read_text(encoding="utf-8"))
print(json.dumps(meta["episode"]["observe_sources"], indent=2)[:800])

## Parte 4 — El presupuesto: la presión

El episodio da un presupuesto finito. Observar es barato; **experimentar es caro** (costo fijo +
por fila). Esa aritmética es el corazón del examen: obliga a *decidir* qué evidencia comprar.
Un mundo con todo gratis no mide juicio (nadie tiene que elegir); uno carísimo tampoco (nadie
puede investigar). La escasez está calibrada para que la jugada viciosa sea la tentadora.

In [ ]:
PRESUPUESTO = 5000.0
EXP_FIJO, EXP_POR_FILA = 100.0, 2.0

gasto = 0.0
_, c = vista_registros(400, seed=3); gasto += c            # 400 filas de registros
gasto += EXP_FIJO + EXP_POR_FILA * 50                       # un experimento de 50 unidades
gasto += EXP_FIJO + EXP_POR_FILA * 50                       # otro
print(f"gastado: {gasto}  |  queda: {PRESUPUESTO - gasto}")
print("(cada experimento de 50 filas cuesta lo que 400 filas historicas)")

## Parte 5 — Dos investigadores (scriptados, sin LLM)

Los dos ven exactamente los mismos datos. La diferencia es **el juicio**.

**El SEDUCIDO**: mira el histórico, corre una regresión, encuentra el coeficiente y se casa
con la primera historia — la que el folklore ya le había susurrado.

In [ ]:
datos, _ = vista_registros(400, seed=3)

# regresion sabor ~ molienda (la historia facil)
X = np.column_stack([np.ones(len(datos)), datos["molienda"]])
beta = np.linalg.lstsq(X, datos["sabor"].values, rcond=None)[0]
print(f"EL SEDUCIDO concluye: sabor = {beta[0]:.1f} + {beta[1]:.2f} * molienda")
print("  'cada punto de molienda vale", round(beta[1], 2), "de sabor — cambien el molinillo!'")

**El CUIDADOSO**: nota que hay timestamps, **estratifica por hora** (la escapatoria barata:
dentro de una ventana angosta de tiempo, el confounder casi no varía) y ve que la correlación se
evapora. Después paga **un experimento** (la escapatoria cara pero definitiva) para confirmar.

In [ ]:
# escapatoria barata: la correlacion DENTRO de cada franja horaria
for lo, hi in [(0.0, 0.2), (0.4, 0.6), (0.8, 1.0)]:
    franja = datos[(datos.hora_norm >= lo) & (datos.hora_norm < hi)]
    print(f"franja hora {lo}-{hi}: corr molienda-sabor = {franja['molienda'].corr(franja['sabor']):+.3f}  (n={len(franja)})")

# escapatoria cara: el experimento do(molinillo)
exp_lo = toy_sample(regime({"molinillo": 0.0}), 50, seed=21)
exp_hi = toy_sample(regime({"molinillo": 10.0}), 50, seed=22)
print(f"\nexperimento: sabor con molinillo=0 -> {exp_lo['sabor'].mean():.2f}   molinillo=10 -> {exp_hi['sabor'].mean():.2f}")
print("EL CUIDADOSO concluye: el molinillo NO mueve el sabor; hay un factor del dia (oculto) que mueve ambos.")

## Parte 6 — La entrega: el contrato

El agente no entrega un ensayo: entrega **un programa** con la misma firma que el mundo —
`model(regime, n, seed) -> tabla` — su *maqueta* del proceso. Todo lo que crea del mundo tiene
que estar ahí adentro, ejecutable. (El contrato exacto y sin ambigüedades es el `DeliverySpec v1`
de [scripts/out/probe_0135/DELIVERY_SPEC_v1.md](../scripts/out/probe_0135/DELIVERY_SPEC_v1.md).)

Las dos entregas, tal como las escribirían nuestros investigadores:

In [ ]:
modelo_seducido = '''
import numpy as np, pandas as pd
def model(regime, n, seed):
    rng = np.random.default_rng(seed)
    if "molinillo" in regime.config:                      # su unica palanca: el molinillo
        molienda = float(regime.config["molinillo"]) - 5 + 6 + rng.normal(0, 0.55, n)
    else:
        molienda = rng.normal(6.0, 1.9, n)                # reproduce el pasado que ajusto
    sabor = 21.2 + 1.44*molienda + rng.normal(0, 2.4, n)  # su regresion, tomada como CAUSA
    return pd.DataFrame({"molienda": molienda, "sabor": sabor})
'''

modelo_cuidadoso = '''
import numpy as np, pandas as pd
def model(regime, n, seed):
    rng = np.random.default_rng(seed)
    cfg = regime.config
    if "descansos" in cfg:
        cansancio = 10.0 - float(cfg["descansos"]) + rng.normal(0, 0.3, n)
    else:
        cansancio = 2.0 + 6.0*rng.uniform(0, 1, n) + rng.normal(0, 0.3, n)
    if "molinillo" in cfg:
        molinillo = np.full(n, float(cfg["molinillo"]))
    else:
        molinillo = np.clip(rng.normal(5, 1, n), 0, 10)
    molienda = 6 + 1.0*(molinillo-5) - 0.8*(cansancio-5) + rng.normal(0, 0.5, n)
    sabor    = 30 - 2.0*(cansancio-5) + rng.normal(0, 1.5, n)
    return pd.DataFrame({"molienda": molienda, "sabor": sabor})
'''
print("dos entregas listas (strings de codigo, igual que entregan los agentes reales)")

## Parte 7 — La batería, la distancia y la nota

**La batería** es la hoja de preguntas del examen: una lista de regímenes con pesos, **oculta
para el agente**. Incluye el histórico Y las intervenciones — ahí es donde el seducido no puede
esconderse: su modelo predice que `do(molinillo=10)` mejora el sabor, y el mundo verdadero dice
que no.

**La nota**: por cada régimen se comparan las distribuciones (verdad vs maqueta) con una
distancia estadística (*energy distance* — la pieza real del repo). Se normaliza contra un
tope `D_MAX` = 1.5× la distancia de un modelo "que no sabe nada" (el ancla): quedar peor que
no-saber-nada da 0. **Cero LLM en todo este camino** — regla dura del proyecto.

In [ ]:
from wager.reward.distance import TruthSide          # la pieza REAL del scoring
from wager.reward.sandbox import SandboxedSubmission  # el sandbox REAL de submissions

COLS = ["molienda", "sabor"]
BATERIA = [  # (peso, regimen) — la hoja de preguntas del examinador
    (1.0, regime()),                      # el proceso historico
    (2.0, regime({"molinillo": 10.0})),   # el counterfactual del folklore (la pregunta del cliente!)
    (2.0, regime({"molinillo": 0.0})),
    (2.0, regime({"descansos": 8.0})),    # la palanca verdadera
    (1.0, regime({"molinillo": 8.0, "descansos": 3.0})),
]
N = 3000

# el ANCLA: un "modelo" que no sabe nada de perillas — la mezcla de todos los regimenes.
# Quedar peor que el ancla da 0. D_MAX = 1.5 x distancia(ancla, verdad), como en produccion.
verdades = [toy_sample(ns, N, seed=100 + i)[COLS] for i, (_, ns) in enumerate(BATERIA)]
pool = pd.concat(verdades, ignore_index=True)

def nota_R(codigo):
    filas, total_peso, total = [], 0.0, 0.0
    rng = np.random.default_rng(999)
    with SandboxedSubmission(codigo, COLS, timeout_s=15.0) as sb:
        for i, (peso, ns) in enumerate(BATERIA):
            ts = TruthSide(verdades[i], COLS)
            d = ts.distance_to(sb.run(ns, N, 200 + i))
            ancla = pool.sample(N, random_state=int(rng.integers(1e6))).reset_index(drop=True)
            d_max = 1.5 * ts.distance_to(ancla)
            item = 1.0 - min(d, d_max) / d_max               # 1 = clavado, 0 = peor que el ancla
            filas.append((ns.config, round(d, 3), round(item, 3)))
            total += peso * item; total_peso += peso
    return total / total_peso, filas

for nombre, codigo in [("SEDUCIDO", modelo_seducido), ("CUIDADOSO", modelo_cuidadoso)]:
    R, filas = nota_R(codigo)
    print(f"\n{nombre}: R = {R:.3f}")
    for cfg, d, item in filas:
        print(f"   regimen {str(cfg):38} distancia={d:7.3f}  item={item:.3f}")

**Leé la tabla**: el seducido clava el histórico (ahí su historia ajusta perfecto — por eso
es seductora) y se **hunde exactamente en los regímenes intervenidos**. El vicio no se castiga
por decreto: es la jugada perdedora del mundo. Eso es "las conductas se observan, las
consecuencias se cobran" — el principio central del scoring.

*(Nota fina: esta batería de juguete incluye a propósito los regímenes que cobran el defecto.
Lo que aprendimos esta semana — el audit de la entrega 0.9866 — es que las baterías reales
necesitan exactamente esto: regímenes que cobren los defectos frecuentes, y una normalización
que no los comprima.)*

## Parte 8 — Producción real: el caso del repo, de punta a punta

Todo lo anterior, empaquetado: `build_world_server` levanta el caso real
(`first_story_scarce_v0`), con su brief, sus fuentes declaradas, su presupuesto, su batería
certificada y su scoring de producción. Vas a ver el mismo flujo: describir → comprar →
experimentar → entregar → nota.

In [ ]:
from wager.harness.case_episode import build_world_server

server = build_world_server(ROOT / "cases" / "first_story_scarce_v0", seed_offset=123)
ficha = server.describe()
print(ficha["brief"][:900])

In [ ]:
obs = server.observe("registros_linea", 100)     # compra real: 100 filas historicas
print(f"quedan {server.budget_remaining:.0f} de presupuesto tras comprar 100 filas")
obs.head()

In [ ]:
from wager.contracts import ExperimentDesign

exp = server.experiment(ExperimentDesign(config={"humidity": 2.0}, n=40))
print(f"experimento do(humidity=2): sabor... digo, outcome medio = {exp['outcome'].mean():.2f}")
print(f"quedan {server.budget_remaining:.0f} de presupuesto")

Y las dos entregas de verdad — la **traducción perfecta de la verdad** (la referencia del
probe de esta semana) y el **folklore** (la creencia de la planta). El server puntúa con el
camino de producción completo (batería certificada, anclas, funcionales, sandbox):

In [ ]:
from probe_0135_build import reference_code, TRUTH_MANIFEST
from exp_probe_0135_stage2 import REF_FOLKLORE

server_a = build_world_server(ROOT / "cases" / "first_story_scarce_v0", seed_offset=123)
server_a.submit(reference_code(TRUTH_MANIFEST))
print(f"entrega VERDAD-EXACTA : R = {server_a.result['R']:.4f}")

server_b = build_world_server(ROOT / "cases" / "first_story_scarce_v0", seed_offset=123)
server_b.submit(REF_FOLKLORE)
print(f"entrega FOLKLORE      : R = {server_b.result['R']:.4f}")

## Cierre — dónde vive cada pieza

| Lo que viste | La pieza real |
|---|---|
| el mundo / la verdad | `cases/<caso>/world.py` (`sample(regime, n, seed)`) |
| regímenes y contratos | `wager/contracts.py` (`Regime`, `Battery`, `ExperimentDesign`…) |
| fuentes, canal, precios | `meta.json` del caso (`episode.observe_sources`, `channel`, `hidden_columns`) |
| el server del episodio | `wager/harness/world_server.py` + `build_world_server` |
| el kernel que usa el agente LLM | `wager/harness/kernel_proc.py` (el agente escribe celdas; `env` = este server) |
| la distancia | `wager/reward/distance.py` (`TruthSide`) |
| el scoring completo | `wager/reward/scorer.py` (batería + anclas + funcionales + MDL, cero-LLM) |
| el sandbox de submissions | `wager/reward/sandbox.py` |
| batería/rivales/certificados | `wager/factory/` (derivados de `meta.json`, nunca a mano) |
| el contrato de entrega v1 | `scripts/out/probe_0135/DELIVERY_SPEC_v1.md` |

**Lo único que este notebook no corre** es el agente LLM real (para no gastar API): en producción,
el agente recibe el brief, escribe celdas Python en el kernel (`env.observe(...)`,
`env.experiment(...)`) y termina con `env.submit(codigo)` — exactamente las llamadas que vos
hiciste a mano en la Parte 8.